# Imports

In [ ]:
import subprocess
import sys
import json
from typing import List, Dict, Optional, Union, Literal
from dataclasses import dataclass, field
from pathlib import Path

# Config

In [ ]:
@dataclass
class GarakConfig:
    """
    Configuration class for Garak testing parameters.

    Attributes:
        model_name: Name of the Ollama model to test (e.g. 'llama2', 'mistral')
        model_type: Type of model interface (default: 'ollama')
        probes: List of probe modules to run (None = all probes)
        output_dir: Directory to store test results
        parallel_requests: Number of parallel requests to make
    """
    model_name: str
    model_type: Literal['ollama'] = 'ollama'
    probes: Optional[List[str]] = None
    output_dir: Path = field(default_factory=lambda: Path('./garak_reports'))
    parallel_requests: int = 1

    def __post_init__(self) -> None:
        """Validate configuration parameters after initialization."""
        if not self.model_name:
            raise ValueError("model_name cannot be empty.")

        if self.parallel_requests < 1:
            raise ValueError("parallel_requests must be at least 1")

        # Ensure output directory exists
        self.output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Configure the test
config = GarakConfig(
    model_name='llama3.2:3b',
    probes=[
        # 'promptinject',     # Test prompt injection vulnerabilities
        'dan',              # Test "Do Anything Now" jailbreak attempts
        # 'encoding',         # Test encoding-based attacks
    ],
    output_dir=Path('./garak_test_results').absolute(),
    parallel_requests=1,
)

# Ollama Validator

In [ ]:
class OllamaModelValidator:
    """Validator for checking Ollama model availability."""

    @staticmethod
    def check_ollama_running() -> bool:
        """
        Check if Ollama servise is running.

        Returns:
            True if Ollama is accessible, False otherwise
        """
        try:
            result = subprocess.run(
                ['ollama', 'list'],
                capture_output=True,
                text=True,
                timeout=5,
                check=False
            )
            return result.returncode == 0

        except (subprocess.SubprocessError, FileNotFoundError) as e:
            print(f"Error checking Ollama: {e}", file=sys.stderr)
            return False

    @staticmethod
    def get_available_models() -> List[str]:
        """
        Retrieve list of available Ollama models.

        Returns:
            List of model names available in Ollama

        Raises:
            RuntimeError: If unable to retrieve model list
        """
        try:
            result = subprocess.run(
                ['ollama', 'list'],
                capture_output=True,
                text=True,
                timeout=10,
                check=True
            )

            # Parse output (skip header line)
            lines = result.stdout.strip().split('\n')[1:]
            models = [line.split()[0] for line in lines if line.strip()]
            return models

        except subprocess.CalledProcessError as e:
            raise RuntimeError(f"Failed to get model list: {e.stderr}") from e

        except subprocess.TimeoutExpired as e:
            raise RuntimeError("Timeout while fetching model list") from e

    @staticmethod
    def validate_model_exists(model_name='str') -> bool:
        """
        Validate that a specific model exists in Ollama.

        Args:
            model_name: Name of the model to validate

        Returns:
            True if model exists, False otherwise
        """
        try:
            available_models = OllamaModelValidator.get_available_models()
            return model_name in available_models

        except RuntimeError:
            return False

# Garak Runner

In [ ]:
class GarakRunner:
    """Main class for executing Garak security tests on Ollama models."""

    def __init__(self, config: GarakConfig) -> None:
        """
        Initialize the Garak runner with configuration.

        Args:
            config: GarakConfig object with test parameters
        """
        self.config = config
        self.validator = OllamaModelValidator()

    def _build_command(self) -> List[str]:
        """
        Build the garak command with all parameters.

        Returns:
            List of command arguments for subprocess
        """
        self.config.output_dir.mkdir(parents=True, exist_ok=True)

        report_prefix = self.config.output_dir.absolute() / 'report'

        cmd = [
            'garak',
            '--target_type', self.config.model_type,
            '--target_name', self.config.model_name,
            '--report_prefix', str(report_prefix),
            '--parallel_requests', str(self.config.parallel_requests),
        ]

        # Add specific probes if configured
        if self.config.probes:
            for probe in self.config.probes:
                cmd.extend(['--probes', probe])

        return cmd

    def validate_environment(self) -> None:
        """
        Validate that the testing environment is properly configured.

        Raises:
            EnvironmentError: If environment validation fails.
        """
        # Check if Ollama is running
        if not self.validator.check_ollama_running():
            raise EnvironmentError(
                "Ollama is not running. Please start Ollama service first.\n"
                "Run: ollama serve"
            )

        # Check if model exists
        if not self.validator.validate_model_exists(self.config.model_name):
            available = self.validator.get_available_models()
            raise EnvironmentError(
                f"Model '{self.config.model_name}' not found.\n"
                f"Available models: {', '.join(available)}\n"
                f"Pull model with: ollama pull {self.config.model_name}"
            )

        # Check if garak is installed
        try:
            subprocess.run(
                ['garak', '--version'],
                capture_output=True,
                check=True,
                timeout=5
            )

        except (subprocess.CalledProcessError, FileNotFoundError) as e:
            raise EnvironmentError(
                "Garak is not installed. Install it with: pip install garak"
            ) from e

    def run_tests(self, verbose: bool = True) -> Dict[str, Union[int, str]]:
        """
        Execute Garak security tests on the configured model.
        
        Args:
            verbose: If True, print command output in real-time
            
        Returns:
            Dictionary containing test results and metadata
            
        Raises:
            RuntimeError: If test execution fails
        """
        print(f"\n{'='*60}")
        print(f"Starting Garak Security Tests")
        print(f"{'='*60}")
        print(f"Model: {self.config.model_name}")
        print(f"Output Directory: {self.config.output_dir}")
        if self.config.probes:
            print(f"Probes: {', '.join(self.config.probes)}")
        else:
            print("Probes: All available probes")
        print(f"{'='*60}\n")
        
        cmd = self._build_command()
        
        try:
            # Run garak with real-time output if verbose
            if verbose:
                process = subprocess.Popen(
                    cmd,
                    stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT,
                    text=True,
                    bufsize=1,
                    universal_newlines=True
                )
                
                # Stream output in real-time
                for line in process.stdout:
                    print(line, end='')
                
                returncode = process.wait()
                
                if returncode != 0:
                    raise RuntimeError(f"Garak tests failed with exit code {returncode}")
            else:
                # Run without real-time output
                result = subprocess.run(
                    cmd,
                    capture_output=True,
                    text=True,
                    check=True,
                    timeout=3600  # 1 hour timeout
                )
            
            print(f"\n{'='*60}")
            print("Tests completed successfully!")
            print(f"Results saved to: {self.config.output_dir}")
            print(f"{'='*60}\n")
            
            return {
                'status': 'success',
                'output_dir': str(self.config.output_dir),
                'model': self.config.model_name
            }
            
        except subprocess.TimeoutExpired as e:
            raise RuntimeError("Test execution timed out after 1 hour") from e
        except subprocess.CalledProcessError as e:
            raise RuntimeError(
                f"Garak execution failed:\n{e.stderr}"
            ) from e
        except Exception as e:
            raise RuntimeError(f"Unexpected error during test execution: {e}") from e

In [ ]:
runner = GarakRunner(config)

In [ ]:
runner.validate_environment()

In [ ]:
# Execute tests
results = runner.run_tests(verbose=True)

# Summary

In [ ]:
print("Test Summary:")
print(json.dumps(results, indent=2))